In [1]:
from utils import extract_chord_path_and_annotations, get_chords_and_annotations, extract_indices_subset
from utils import get_idf_dict

In [2]:
import numpy as np
import math
from collections import defaultdict
from utils import split_dict, extract_n_grams
from rule_based import get_uni_gram, train_all, RuleModel, evaluate_test_set
import dill

In [3]:
cols2extract = ['chord_locations','CFP','CFD',
                'UC','RHC','CPT',
                'BD','R','weighted_total']


files_and_annotations = extract_chord_path_and_annotations("./data/Annotations.csv", cols2extract, delimiter=",")
chords_and_annotations = get_chords_and_annotations(files_and_annotations, no_duplicates=True)

## Generate 10 random splits for the k-fold

In [4]:
s= list(range(200))

In [ ]:
random.shuffle(s)
split_size = 20
split_1_ind = s[-split_size:]
del s[-split_size:]
split_2_ind = s[-split_size:]
del s[-split_size:]
split_3_ind = s[-split_size:]
del s[-split_size:]
split_4_ind = s[-split_size:]
del s[-split_size:]
split_5_ind = s[-split_size:]
del s[-split_size:]
split_6_ind = s[-split_size:]
del s[-split_size:]
split_7_ind = s[-split_size:]
del s[-split_size:]
split_8_ind = s[-split_size:]
del s[-split_size:]
split_9_ind = s[-split_size:]
del s[-split_size:]
split_10_ind = s[-split_size:]
del s[-split_size:]


In [ ]:
# print the splits to copy and fixate them
print(split_1_ind)
print(split_2_ind)
print(split_3_ind)
print(split_4_ind)
print(split_5_ind)
print(split_6_ind)
print(split_7_ind)
print(split_8_ind)
print(split_9_ind)
print(split_10_ind)


In [5]:
## final indices
split_1_ind =[88, 71, 4, 171, 33, 56, 105, 72, 139, 64, 26, 117, 151, 183, 80, 153, 121, 63, 176, 97]
split_2_ind =[132, 182, 3, 168, 39, 165, 147, 109, 114, 0, 177, 133, 160, 198, 20, 75, 32, 123, 157, 193]
split_3_ind =[167, 57, 100, 195, 27, 118, 128, 190, 103, 141, 162, 186, 13, 81, 55, 8, 36, 135, 50, 2]
split_4_ind =[172, 79, 10, 148, 73, 92, 164, 34, 40, 47, 11, 25, 98, 85, 136, 5, 142, 99, 161, 126]
split_5_ind =[102, 62, 23, 74, 178, 170, 112, 24, 106, 95, 152, 90, 101, 138, 21, 197, 7, 150, 77, 42]
split_6_ind =[35, 187, 52, 194, 115, 16, 116, 6, 53, 45, 15, 67, 131, 185, 191, 1, 96, 28, 19, 66]
split_7_ind =[83, 9, 59, 175, 76, 22, 51, 69, 14, 94, 38, 46, 154, 91, 104, 173, 93, 108, 48, 12]
split_8_ind =[180, 86, 137, 119, 43, 120, 84, 49, 58, 18, 89, 179, 155, 166, 111, 129, 192, 143, 134, 124]
split_9_ind =[17, 159, 82, 61, 41, 54, 184, 130, 199, 188, 181, 78, 60, 169, 174, 37, 29, 140, 145, 163]
split_10_ind =[144, 87, 70, 127, 68, 44, 196, 158, 31, 125, 113, 146, 122, 65, 156, 30, 149, 189, 107, 110]
split_dict = {
    1: split_1_ind,
    2: split_2_ind,
    3: split_3_ind,
    4: split_4_ind,
    5: split_5_ind,
    6: split_6_ind,
    7: split_7_ind,
    8: split_8_ind,
    9: split_9_ind,
    0: split_10_ind,
}

## Do k-fold cross validation for rule based

In [6]:
save_performance = []
for i in range(10):
    # obtain train-validate-test-splits
    indices = np.asarray(list(range(10))) + i
    
    train_ind = [split_dict[(m% 10)] for m in indices[:-1]]
    validate_ind = [split_dict[(m % 10)] for m in indices[-1:]]

    train_ind = [k for j in train_ind for k in j]
    validate_ind = [k for j in validate_ind for k in j]
    
    # obtain chords, lab_files and rest of data
    train = extract_indices_subset(train_ind, chords_and_annotations)
    validate = extract_indices_subset(validate_ind, chords_and_annotations)
    
    # obtain idf and gram dicts
    train_idf_dict = get_idf_dict(train, n_gram=1, simplified=False)
    train_idf_dict = {k:math.log(200/v) for k,v in train_idf_dict.items()}
    train_idf_dict = defaultdict(lambda:max(train_idf_dict.values()),train_idf_dict)
    train_uni_grams = get_uni_gram(train)
    
    # train rule based model
    cfg = train_all(train, train_idf_dict, train_uni_grams)
    
    # create trained model
    kfold_model = RuleModel(cfg, train_uni_grams,  train_idf_dict)
    with open(f'./rule_models/fold_{i}.pkl','wb') as f:
        dill.dump(kfold_model, f)
    with open(f'./rule_models/idf_dict_{i}.pkl','wb') as f:
        dill.dump(kfold_model.idf_dict, f)
    
    # evaluate for 
    all_performance = evaluate_test_set(kfold_model, validate, classify=False, custom_loss=True)
    save_performance.append(all_performance)

(1, 2, 1, 1, 2, 3, 0) 4.228167825645064 1 1
(1, 1, 1, 1, 1, 0, 0) 2.7231192107921247 1 1
(0, 1, 0, 1, 2, 3, 1) 3.3784209061786274 1 0
(3, 2, 2, 1, 2, 3, 1) 4.453406595505497 1 3
(1, 2, 1, 2, 2, 2, 1) 3.636539950745699 1 1
(1, 1, 1, 1, 2, 2, 1) 9.89159458354331 1 1
(2, 2, 1, 1, 3, 0, 1) 3.6128981458733325 1 2
(0, 1, 1, 1, 2, 2, 1) 2.255278113210986 1 0
(0, 1, 1, 1, 1, 1, 0) 0.9555462974358317 1 0
(2, 3, 2, 1, 2, 1, 1) 10.11963843589186 1 2
(0, 1, 1, 1, 2, 1, 0) 2.802963682987427 1 0
(3, 3, 3, 2, 2, 3, 2) 10.810536957741986 1 3
(1, 2, 2, 2, 2, 3, 0) 4.090168491894072 1 1
(0, 1, 0, 3, 1, 0, 1) 1.4789091676237114 1 0
(0, 1, 1, 1, 1, 0, 1) 6.560039264828983 1 0
(0, 1, 0, 1, 3, 0, 0) 2.8309963427048204 1 0
(0, 2, 0, 1, 2, 0, 1) 3.3872862979448155 1 0
(1, 1, 1, 1, 1, 0, 1) 5.146319334847059 1 1
(0, 1, 0, 1, 2, 3, 1) 1.7182574179017773 1 0
(1, 1, 2, 2, 2, 3, 1) 4.351715080554934 1 1
(2, 2, 2, 2, 2, 2, 1) 7.47139833434324 1 2
(2, 2, 2, 2, 3, 0, 1) 4.463720731165792 1 2
(0, 1, 0, 1, 1, 0, 0) 1.5

(1, 2, 1, 1, 2, 3, 0) 4.228167825645064 1 1
(1, 1, 1, 1, 1, 0, 0) 3.6249604646726414 1 1
(0, 1, 0, 1, 2, 3, 1) 3.3784209061786274 1 0
(3, 2, 2, 1, 2, 3, 1) 4.970315606876037 1 3
(1, 2, 1, 2, 2, 2, 1) 3.636539950745699 1 1
(1, 1, 1, 1, 2, 2, 1) 9.89159458354331 1 1
(2, 2, 1, 1, 3, 0, 1) 3.6128981458733325 1 2
(0, 1, 1, 1, 2, 2, 1) 2.255278113210986 1 0
(0, 1, 1, 1, 1, 1, 0) 1.1480095733026459 1 0
(2, 3, 2, 1, 2, 1, 1) 13.608774262643008 1 2
(0, 1, 1, 1, 2, 1, 0) 2.802963682987427 1 0
(3, 3, 3, 2, 2, 3, 2) 17.992322559988605 1 3
(1, 2, 2, 2, 2, 3, 0) 7.0212477563976785 1 1
(0, 1, 0, 3, 1, 0, 1) 1.4789091676237114 1 0
(0, 1, 1, 1, 1, 0, 1) 8.81676551911045 1 0
(0, 1, 0, 1, 3, 0, 0) 2.8309963427048204 1 0
(0, 2, 0, 1, 2, 0, 1) 3.3872862979448155 1 0
(1, 1, 1, 1, 1, 0, 1) 5.146319334847059 1 1
(0, 1, 0, 1, 2, 3, 1) 1.7182574179017773 1 0
(1, 1, 2, 2, 2, 3, 1) 5.624820435278415 1 1
(2, 2, 2, 2, 2, 2, 1) 10.21348500914915 1 2
(2, 2, 2, 2, 3, 0, 1) 4.715388451188559 1 2
(0, 1, 0, 1, 1, 0, 0) 1

(1, 2, 1, 1, 2, 3, 0) 3.577487923746801 2 2
(1, 1, 1, 1, 1, 0, 0) 1.6156620637748642 2 1
(0, 1, 0, 1, 2, 3, 1) 2.0999661376504233 2 1
(3, 2, 2, 1, 2, 3, 1) 4.042304630353774 2 2
(1, 2, 1, 2, 2, 2, 1) 1.7055912506727837 2 2
(1, 1, 1, 1, 2, 2, 1) 7.5279465325353385 2 1
(2, 2, 1, 1, 3, 0, 1) 2.749062144366518 2 2
(0, 1, 1, 1, 2, 2, 1) 1.8380062365923293 2 1
(0, 1, 1, 1, 1, 1, 0) 1.3058926792144963 2 1
(2, 3, 2, 1, 2, 1, 1) 8.592076855587324 2 3
(0, 1, 1, 1, 2, 1, 0) 2.455992290357341 2 1
(3, 3, 3, 2, 2, 3, 2) 8.399233568157115 2 3
(1, 2, 2, 2, 2, 3, 0) 3.1808306085773856 2 2
(0, 1, 0, 3, 1, 0, 1) 1.3631334755734223 2 1
(0, 1, 1, 1, 1, 0, 1) 4.990045992729365 2 1
(0, 1, 0, 1, 3, 0, 0) 2.6817432787520636 2 1
(0, 2, 0, 1, 2, 0, 1) 2.42593327417103 2 2
(1, 1, 1, 1, 1, 0, 1) 4.414942251797957 2 1
(0, 1, 0, 1, 2, 3, 1) 1.2709085454612612 2 1
(1, 1, 2, 2, 2, 3, 1) 2.718113008776973 2 1
(2, 2, 2, 2, 2, 2, 1) 6.169265687743153 2 2
(2, 2, 2, 2, 3, 0, 1) 2.9069818813566894 2 2
(0, 1, 0, 1, 1, 0, 0) 

(1, 2, 1, 1, 2, 3, 0) 6.943781040577983 2 2
(1, 1, 1, 1, 1, 0, 0) 4.846986191324593 2 1
(0, 1, 0, 1, 2, 3, 1) 4.779197341182155 2 1
(3, 2, 2, 1, 2, 3, 1) 9.363580398720979 2 2
(1, 2, 1, 2, 2, 2, 1) 5.116773752018352 2 2
(1, 1, 1, 1, 2, 2, 1) 14.27384499445234 2 1
(2, 2, 1, 1, 3, 0, 1) 6.090852376183251 2 2
(0, 1, 1, 1, 2, 2, 1) 4.456111087490877 2 1
(0, 1, 1, 1, 1, 1, 0) 3.9176780376434888 2 1
(2, 3, 2, 1, 2, 1, 1) 16.775867090090834 2 3
(0, 1, 1, 1, 2, 1, 0) 5.401699095903042 2 1
(3, 3, 3, 2, 2, 3, 2) 17.4903264557968 2 3
(1, 2, 2, 2, 2, 3, 0) 8.998897721162413 2 2
(0, 1, 0, 3, 1, 0, 1) 4.006146345486756 2 1
(0, 1, 1, 1, 1, 0, 1) 10.058777369586908 2 1
(0, 1, 0, 1, 3, 0, 0) 5.3387313010555575 2 1
(0, 2, 0, 1, 2, 0, 1) 5.235999639581736 2 2
(1, 1, 1, 1, 1, 0, 1) 9.63403100627412 2 1
(0, 1, 0, 1, 2, 3, 1) 3.2382409182896303 2 1
(1, 1, 2, 2, 2, 3, 1) 6.748770136302 2 1
(2, 2, 2, 2, 2, 2, 1) 11.875426733482582 2 2
(2, 2, 2, 2, 3, 0, 1) 7.184004530976521 2 2
(0, 1, 0, 1, 1, 0, 0) 2.3983063

(1, 2, 1, 1, 2, 3, 0) 7.154975847493602 2 2
(1, 1, 1, 1, 1, 0, 0) 3.2313241275497284 2 1
(0, 1, 0, 1, 2, 3, 1) 4.199932275300847 2 1
(3, 2, 2, 1, 2, 3, 1) 8.084609260707548 2 2
(1, 2, 1, 2, 2, 2, 1) 3.4111825013455674 2 2
(1, 1, 1, 1, 2, 2, 1) 15.055893065070677 2 1
(2, 2, 1, 1, 3, 0, 1) 5.498124288733036 2 2
(0, 1, 1, 1, 2, 2, 1) 3.6760124731846586 2 1
(0, 1, 1, 1, 1, 1, 0) 2.6117853584289925 2 1
(2, 3, 2, 1, 2, 1, 1) 17.18415371117465 2 3
(0, 1, 1, 1, 2, 1, 0) 4.911984580714682 2 1
(3, 3, 3, 2, 2, 3, 2) 16.79846713631423 2 3
(1, 2, 2, 2, 2, 3, 0) 6.361661217154771 2 2
(0, 1, 0, 3, 1, 0, 1) 2.7262669511468447 2 1
(0, 1, 1, 1, 1, 0, 1) 9.98009198545873 2 1
(0, 1, 0, 1, 3, 0, 0) 5.363486557504127 2 1
(0, 2, 0, 1, 2, 0, 1) 4.85186654834206 2 2
(1, 1, 1, 1, 1, 0, 1) 8.829884503595913 2 1
(0, 1, 0, 1, 2, 3, 1) 2.5418170909225224 2 1
(1, 1, 2, 2, 2, 3, 1) 5.436226017553946 2 1
(2, 2, 2, 2, 2, 2, 1) 12.338531375486307 2 2
(2, 2, 2, 2, 3, 0, 1) 5.813963762713379 2 2
(0, 1, 0, 1, 1, 0, 0) 1.67

(1, 2, 1, 1, 2, 3, 0) 7.3661706544092205 2 2
(1, 1, 1, 1, 1, 0, 0) 1.6156620637748642 2 1
(0, 1, 0, 1, 2, 3, 1) 3.620667209419538 2 1
(3, 2, 2, 1, 2, 3, 1) 6.8056381226941145 2 2
(1, 2, 1, 2, 2, 2, 1) 1.7055912506727837 2 2
(1, 1, 1, 1, 2, 2, 1) 15.837941135689022 2 1
(2, 2, 1, 1, 3, 0, 1) 4.90539620128282 2 2
(0, 1, 1, 1, 2, 2, 1) 2.8959138588784397 2 1
(0, 1, 1, 1, 1, 1, 0) 1.3058926792144963 2 1
(2, 3, 2, 1, 2, 1, 1) 17.59244033225846 2 3
(0, 1, 1, 1, 2, 1, 0) 4.422270065526321 2 1
(3, 3, 3, 2, 2, 3, 2) 16.106607816831662 2 3
(1, 2, 2, 2, 2, 3, 0) 3.724424713147131 2 2
(0, 1, 0, 3, 1, 0, 1) 1.4463875568069333 2 1
(0, 1, 1, 1, 1, 0, 1) 9.90140660133055 2 1
(0, 1, 0, 1, 3, 0, 0) 5.388241813952697 2 1
(0, 2, 0, 1, 2, 0, 1) 4.4677334571023835 2 2
(1, 1, 1, 1, 1, 0, 1) 8.02573800091771 2 1
(0, 1, 0, 1, 2, 3, 1) 1.8453932635554147 2 1
(1, 1, 2, 2, 2, 3, 1) 4.123681898805894 2 1
(2, 2, 2, 2, 2, 2, 1) 12.801636017490033 2 2
(2, 2, 2, 2, 3, 0, 1) 4.443922994450236 2 2
(0, 1, 0, 1, 1, 0, 0) 0

... (563 outputs left out)

(0, 2, 0, 1, 1, 0, 0) 0.4951814058749733 5 1
(0, 2, 0, 1, 1, 0, 0) 0.927511337999988 5 1
(0, 2, 1, 1, 0, 0, 0) 1.023472222124937 5 0
(0, 1, 0, 0, 0, 0, 0) 1.9893242629997019 5 0
(2, 3, 2, 1, 1, 0, 2) 0.8049546485000292 5 1
(0, 2, 1, 1, 1, 0, 0) 1.380194197333367 5 1
(2, 3, 2, 2, 2, 0, 2) 0.5955002834374454 5 2
(0, 2, 1, 1, 0, 0, 0) 0.8210402494375018 5 0
(0, 2, 1, 1, 1, 0, 1) 0.8291411564999862 5 1
(0, 2, 1, 1, 1, 0, 1) 0.6481533446874863 5 1
(0, 2, 1, 1, 2, 0, 1) 0.7130120936666628 5 2
(0, 2, 1, 1, 1, 0, 1) 0.7326643990834896 5 1
(1, 2, 1, 1, 1, 0, 1) 0.43052295918751327 5 1
(1, 2, 1, 1, 1, 0, 1) 1.758679138499872 5 1
(2, 2, 2, 1, 1, 0, 1) 1.1063718822499595 5 1
(1, 1, 0, 1, 2, 0, 0) 0.7231220909473639 5 2
(1, 2, 1, 1, 2, 0, 1) 0.7428741496242992 5 2
(1, 2, 1, 1, 2, 0, 1) 0.8161196144996268 5 2
(1, 2, 0, 1, 2, 0, 1) 0.3538662131666399 5 2
(1, 2, 1, 1, 1, 0, 1) 0.45769416100000626 5 1
(0, 1, 1, 1, 1, 0, 0) 1.0700736961250072 5 1
(0, 0, 0, 1, 0, 0, 0) 0.823061224499952 5 0
(0, 1, 1, 1, 

(0, 2, 1, 1, 3, 1, 1) 1.0736555561230952 5 3
(2, 2, 2, 2, 2, 3, 1) 2.2625052316311347 5 2
(0, 2, 1, 2, 1, 1, 1) 4.960435944484971 5 1
(1, 1, 1, 3, 3, 3, 1) 1.594198370422256 5 3
(0, 2, 1, 1, 2, 3, 1) 2.3501574931415505 5 2
(0, 2, 0, 1, 2, 0, 0) 2.9213130603962765 5 2
(1, 2, 2, 2, 2, 2, 1) 4.843359306315506 5 2
(1, 2, 1, 1, 2, 0, 1) 1.9060684503166176 5 2
(0, 1, 0, 0, 1, 3, 0) 2.27608056342451 5 1
(1, 2, 1, 1, 2, 0, 0) 3.0242488692240794 5 2
(0, 1, 0, 1, 2, 2, 1) 1.3051179301208504 5 2
(1, 2, 2, 2, 1, 0, 1) 5.205688115835537 5 1
(0, 2, 1, 1, 2, 3, 1) 3.691515370703991 5 2
(1, 2, 2, 3, 2, 0, 1) 6.643466149996016 5 2
(0, 0, 0, 1, 1, 0, 0) 1.3181872749056673 5 1
(1, 3, 2, 2, 2, 0, 0) 8.433402512686364 5 2
(1, 2, 1, 2, 1, 0, 1) 3.6105948079807337 5 1
(1, 2, 1, 2, 2, 1, 1) 1.5431695999222064 5 2
(1, 2, 1, 2, 2, 2, 1) 0.9978990232560118 5 2
(0, 0, 0, 0, 1, 0, 0) 2.3968527949387664 5 1
(0, 1, 0, 1, 1, 0, 0) 2.174781924217137 5 1
(1, 1, 1, 1, 1, 0, 1) 1.9259201417785246 5 1
(2, 2, 2, 1, 2, 0, 1

(0, 2, 1, 1, 1, 0, 0) 8.72121342917605 5 1
(0, 2, 0, 1, 1, 1, 1) 3.7135116512215713 5 1
(0, 1, 0, 1, 2, 0, 0) 0.7740442304094041 5 2
(2, 3, 3, 2, 1, 0, 1) 23.292133903386798 5 1
(0, 1, 1, 1, 1, 1, 1) 0.8653898876233296 5 1
(1, 2, 3, 2, 1, 0, 1) 8.83008922548737 5 1
(0, 1, 1, 1, 1, 0, 1) 2.9819872618738916 5 1
(0, 2, 0, 1, 0, 0, 0) 5.353599030404141 5 0
(0, 1, 1, 1, 1, 1, 1) 4.130654323788534 5 1
(3, 3, 3, 2, 1, 1, 1) 13.036545945057938 5 1
(0, 1, 1, 1, 2, 2, 0) 1.727180631121458 5 2
(0, 0, 0, 1, 0, 0, 0) 0.2983622806573625 5 0
(0, 2, 0, 1, 0, 0, 0) 5.369604401829963 5 0
(2, 2, 2, 2, 2, 2, 1) 8.927785223050975 5 2
(2, 2, 2, 2, 2, 3, 1) 2.836275206971206 5 2
(0, 2, 0, 1, 0, 1, 0) 0.4665222542482785 5 0
(1, 2, 3, 1, 1, 3, 2) 19.612417252211266 5 1
(0, 2, 1, 1, 1, 0, 1) 2.3653429072242247 5 1
(0, 2, 1, 2, 1, 0, 1) 1.9456181878495447 5 1
(0, 0, 1, 1, 1, 0, 0) 0.336502856760903 5 1
(0, 2, 1, 1, 1, 1, 1) 2.707839537395776 5 1
(0, 2, 1, 1, 2, 2, 1) 0.4309589955381436 5 2
(0, 1, 0, 1, 2, 3, 0) 

(1, 2, 2, 2, 2, 0, 1) 0.16521739130434782 6 0
(0, 1, 0, 1, 2, 1, 1) 0.6363636363636364 6 1
(0, 1, 0, 1, 1, 0, 0) 0.05714285714285714 6 0
(1, 2, 2, 1, 2, 2, 1) 0.312 6 2
(0, 2, 1, 1, 1, 0, 1) 0.48 6 0
(1, 3, 2, 2, 1, 0, 0) 0.0 6 0
(1, 2, 2, 1, 1, 0, 1) 0.14754098360655737 6 0
(1, 1, 0, 1, 0, 0, 0) 0.345679012345679 6 0
(1, 2, 2, 1, 1, 0, 1) 0.0 6 0
(2, 2, 2, 2, 1, 2, 0) 0.3611111111111111 6 2
(0, 2, 1, 1, 2, 1, 0) 0.2261904761904762 6 1
(1, 2, 1, 1, 1, 3, 1) 0.04861111111111111 6 3
(0, 2, 1, 1, 1, 0, 0) 0.07272727272727272 6 0
(0, 2, 0, 1, 1, 1, 0) 0.6086956521739131 6 1
(0, 1, 1, 1, 1, 1, 1) 0.39622641509433965 6 1
(0, 2, 1, 1, 0, 0, 0) 0.6027397260273972 6 0
(2, 2, 2, 1, 3, 2, 1) 0.15492957746478872 6 2
(2, 2, 2, 1, 1, 0, 1) 0.8805970149253731 6 0
(0, 2, 1, 1, 2, 3, 1) 0.1917808219178082 6 3
(1, 2, 2, 1, 1, 0, 1) 0.7480916030534351 6 0
(1, 2, 2, 1, 1, 1, 1) 0.6176470588235294 6 1
(0, 2, 0, 1, 1, 0, 0) 0.8322147651006712 6 0
(0, 2, 0, 1, 1, 0, 1) 0.18055555555555555 6 0
(1, 2, 1, 1, 2,

## Do k-fold cross validation for length-based

In [ ]:
def test_len(chords_and_annos, only_len=False, type_of_len='char'):
    cat_scores = {
        1: {1: [], 2: [], 3: [], 4: []},
        2: {1: [], 2: [], 3: [], 4: []},
        3: {1: [], 2: [], 3: [], 4: []},
        4: {1: [], 2: [], 3: [], 4: []},
        5: {1: [], 2: [], 3: [], 4: []},
        6: {1: [], 2: [], 3: [], 4: []},
        7: {1: [], 2: [], 3: [], 4: []},
    }
    weigthed_uni_gram_per_song = []

    lens = []
    
    for (chords, lab_file, sub_scores, total) in chords_and_annos:
        total_chord_count, song_score = 0, 0
        
        long_string = process_song(chords)
        long_string_nl = process_song(chords, remove_nl=False)
        annotations = long_string_nl.replace('\n', ' \n ').replace('  ', ' ').replace('||', '|')
        if type_of_len == 'char':
            cat_scores = len(annotations)
        elif type_of_len == 'dotsplit':
            new_anno = []
            for cho in annotations.split(' '):
                if cho.find(':') != -1:
                    new_anno.append(cho.split(':')[0])
                    new_anno.append(cho.split(':')[1])
                else:
                    new_anno.append(cho)
            cat_scores = len(new_anno)
        elif type_of_len == 'guitar':
            annotations = annotations.split(' ')
            try:
                annotations.remove('')
            except:
                pass
            new_anno = ''
            for ann in annotations:
                if ann in ['|', '\n', 'N', '*']:
                    new_anno += ann + ' '
                else:
                    converted = convert2chordify(ann)
                    to_add = all_chords[converted[0]]['guitar']
                    new_anno += str(to_add)
            cat_scores= len(new_anno)

        if only_len:
            lens.append(cat_scores)

        for i in range(1, 8):
            cat_scores[i][sub_scores[i-1]].append(cat_scores)  
    
    if only_len:
        return lens
    return cat_scores

## add rule-based scores

In [ ]:
def rule_based_custom_data(seq_data, lab_data):
    models = []
    all_songs_classified = []
    for i in range(10):
        with open(f'./rule_models/fold_{i}.pkl', 'rb') as f:
            loaded_model = dill.load(f)
        with open(f'./rule_models/idf_dict_{i}.pkl','rb') as f:
            load_model_cat_dict = dill.load(f)
        loaded_model.idf_dict = load_model_cat_dict
        models.append(loaded_model)
                
    for chord, lab in zip(seq_data,lab_data):
        song_stats = []
        for model in models:
            song_stats.append(model.predict(chord, lab))
#             print(model.predict(chord, lab))
        all_songs_classified.append(song_stats)
#         print()
    return all_songs_classified


In [ ]:
all_the_outputs = rule_based_custom_data(only_seq, only_lab)